<a href="https://colab.research.google.com/github/R341CUL/python-together/blob/main/10-projects/05_tiny_language_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project · A tiny language model (your own baby "LLM")

ChatGPT and friends are **language models**: given some text, predict what comes
next. That's the entire idea — they're just scaled up to billions of parameters
and trained on the internet. Here you'll build a **character-level** language
model small enough to run in a second, watch it babble nonsense, then learn to
spell real-looking words. Same principle, a billion-fold smaller.

Pure NumPy + Python. No GPU, no downloads.

In [ ]:
import numpy as np

## 1. The training text

A real LLM reads the internet. Ours reads a few sentences. Swap in your own text
later — song lyrics, Anny's finance notes, anything.

In [ ]:
text = (
    "sustainable finance blends returns with responsibility. "
    "machine learning finds patterns in data. "
    "python makes both of these fun to learn together. "
) * 40   # repeat so there's enough to learn from

chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}   # char -> number
itos = {i: c for c, i in stoi.items()}       # number -> char
vocab = len(chars)
print("vocabulary size:", vocab, "unique characters")

## 2. The model: "given the last few characters, what's next?"

This is a **bigram-plus** count model — the simplest language model there is. We
count, for each context of `n` previous characters, how often each next
character appears. Predicting = sampling from those counts. (A neural LLM
replaces this count table with a learned network, but the *task* is identical.)

In [ ]:
context_len = 3   # how many previous characters we look at

from collections import defaultdict, Counter
counts = defaultdict(Counter)
for i in range(len(text) - context_len):
    context = text[i:i + context_len]
    nxt = text[i + context_len]
    counts[context][nxt] += 1

print("after 'fin', the model most expects:", counts["fin"].most_common(3))

## 3. Generate text by sampling one character at a time

Start from a seed, predict the next char from the counts, append it, slide the
window, repeat. This "autoregressive" loop is exactly how ChatGPT writes, token
by token.

In [ ]:
def generate(seed, length=200, temperature=1.0, rng=np.random.default_rng(0)):
    out = seed
    for _ in range(length):
        context = out[-context_len:]
        next_counts = counts.get(context)
        if not next_counts:                      # unseen context: restart-ish
            context = out[-1:] * context_len
            next_counts = counts.get(context, Counter({" ": 1}))
        options = list(next_counts.keys())
        weights = np.array(list(next_counts.values()), dtype=float)
        weights = weights ** (1 / temperature)   # temperature: >1 wilder, <1 safer
        probs = weights / weights.sum()
        out += rng.choice(options, p=probs)
    return out

print(generate("sustainable fin", length=180))

Read that output. It isn't random letters — it's learned that `q` is rare, that
spaces follow words, that `tion` and `ing` happen. With `context_len=3` and tiny
data it's charmingly dumb. That gap between "babble" and "fluent" is exactly what
more context, more data, and a neural network (instead of counts) buy you.

## 🏋️ Make it yours

- Increase `context_len` to 5 or 6 and regenerate — more coherent, but it needs
  more text or it just memorizes.
- Change `temperature`: 0.5 (cautious, repetitive) vs 1.5 (wild, creative).
- Replace `text` with your own corpus and watch it pick up *your* style.
- **Stretch:** swap the count table for a small neural network that takes the
  one-hot context and outputs next-char probabilities — reuse the `NeuralNet`
  from `04_build_your_own_neural_net.py`. That's a genuine neural language model.

In [ ]:
# your experiments here
# print(generate("machine ", length=150, temperature=0.7))

### ✅ What you learned
- A language model predicts the **next token** from the previous ones.
- Generation is an **autoregressive loop**: predict, append, repeat.
- **Temperature** controls how adventurous the sampling is.
- Real LLMs are this exact task with a neural network and vastly more scale.